## **Static Context**

In [1]:
from dataclasses import dataclass
@dataclass
class ColorContext:
    favorite_color: str = "blue"
    latest_favorite_color: str = "Yellow"

In [2]:
from langchain.agents import create_agent
agent = create_agent(
    model="gpt-5-nano",
    context_schema=ColorContext
)

In [3]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColorContext()
)

In [4]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='0b773972-2b06-4061-91ec-19bec4505804'),
              AIMessage(content='I don’t know your favourite colour yet. I don’t have access to your preferences. If you’d like, we can figure it out together. Here are a couple of quick options:\n\n- Quick guess game: Answer a few questions and I’ll guess your top color.\n  1) Warm or cool colors?\n  2) Bright/vibrant or soft/pastel?\n  3) Light or dark shade?\n  4) Nature-inspired (blue/green/brown) or bold (red/purple/yellow)?\n  5) Any color you definitely dislike?\n\n- Tell me a few colors you like, and I’ll infer your likely favourite.\n\n- Or say “surprise me” and I’ll pick a color for you and explain why it might fit.\n\nIf you want to start, pick one option and we’ll go from there.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1592, 'prompt_tokens': 12, 'total_tokens': 1604, '

## **Accessing Context**

- context schema is immutable

In [5]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user."""

    return runtime.context.favorite_color

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the latest favourite colour of the user"""
    return runtime.context.latest_favorite_color

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColorContext
)

In [7]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColorContext()
)

/home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=ColorContext(favorite_col...favorite_color='Yellow'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


In [8]:
pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='c1fdcce0-c225-413b-9bc3-057943f7ba85'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_NfIUwB2c2JNR70xQwCwX5TLk', 'function': {'arguments': '{}', 'name': 'get_favourite_colour'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 213, 'prompt_tokens': 149, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CqiohKAPc7Yc7nXUTxIROfEJ8rQRN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b566a-47ca-78a3-b1ba-5a84f5437e40-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_NfIU

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColorContext(favorite_color="black")
)

pprint(response)

/home/md-al-amin/My-Projects/End-to-End-Agentic-Ai-Automation-Lab/.venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=ColorContext(favorite_col...favorite_color='Yellow'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='b1f9b14c-9207-4c07-9e96-dcb522afc140'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_190N5KtOofAeNzrn5hyrJFrb', 'function': {'arguments': '{}', 'name': 'get_favourite_colour'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 149, 'total_tokens': 298, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CqiqG7VHbBFRUrL3PCYhqOFydMkwG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b566b-c179-7f01-8b48-3f31da8c14bb-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_190N

## **Write the state**

In [10]:
from langchain.agents import AgentState
class CustomState(AgentState):
    favourite_colour: str

In [ ]:
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    pass